# 🔍 IQueue — Evaluate XLM-RoBERTa Intent Classifier

Runs the trained model on the test set and computes:
- Overall accuracy
- Per-language accuracy (English, Filipino, Bahasa, Vietnamese)
- Per-intent recall (5 intents)
- Per-language confusion matrices

### Before Running
1. **Upload** the model files (from `xlm-roberta-iqueue.zip`, unzipped)
2. **Upload** `iqueue_test.csv`

In [ ]:
# ============================================================================
# 0. Environment Setup
# ============================================================================

import sys
from pathlib import Path

IS_COLAB = "google.colab" in sys.modules
IS_KAGGLE = "kaggle" in sys.modules if hasattr(sys, "modules") else False

print(f"Environment: {'Colab' if IS_COLAB else 'Kaggle' if IS_KAGGLE else 'Local'}")

In [ ]:
# Install dependencies
!pip install -q transformers scikit-learn pandas numpy

In [ ]:
# ============================================================================
# 1. Upload Files
# ============================================================================

# Edit these paths for your setup
MODEL_DIR = Path("./xlm-roberta-iqueue")   # model files (config.json, model.safetensors, …)
TEST_CSV = Path("./iqueue_test.csv")        # test set

if IS_COLAB:
    from google.colab import files
    print("📎 Upload all model files + iqueue_test.csv:")
    uploaded = files.upload()
    # Find model dir — look for config.json
    for name in uploaded:
        if name.endswith("config.json"):
            # Flatten: model files are in the root
            MODEL_DIR = Path(".")
            break
elif IS_KAGGLE:
    # On Kaggle, model may come from a previous notebook output
    # or be uploaded as a dataset
    print("📁 Kaggle mode")
    if not MODEL_DIR.exists() or not (MODEL_DIR / "config.json").exists():
        # Try common locations
        for candidate in [
            Path("./xlm-roberta-iqueue"),
            Path("/kaggle/input/xlm-roberta-iqueue/xlm-roberta-iqueue"),
            Path("."),  # files in root
        ]:
            if (candidate / "config.json").exists():
                MODEL_DIR = candidate
                break

print(f"Model dir : {MODEL_DIR}  (exists: {MODEL_DIR.exists()})")
print(f"Test CSV  : {TEST_CSV}  (exists: {TEST_CSV.exists()})")

if (MODEL_DIR / "config.json").exists():
    print(f"Model files: {sorted(f.name for f in MODEL_DIR.iterdir() if not f.name.startswith('.'))}")
else:
    print("⚠ config.json not found — did you upload all model files?")

In [ ]:
# ============================================================================
# 2. Imports & Configuration
# ============================================================================

import json
import numpy as np
import pandas as pd
import torch
from collections import defaultdict
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    recall_score,
)
from transformers import pipeline

# ---------------------------------------------------------------------------
# Thresholds
# ---------------------------------------------------------------------------

RECALL_THRESHOLD = 0.70
ACCURACY_THRESHOLD = 0.80

LANGS_FULL = {
    "en": "English",
    "fil": "Filipino",
    "id": "Bahasa Indonesia",
    "vi": "Vietnamese",
}

LABELS = [
    "check_booking",
    "request_requeue",
    "get_departure_info",
    "surge_info",
    "fallback",
]

GPU_AVAILABLE = torch.cuda.is_available()
DEVICE = 0 if GPU_AVAILABLE else -1

print(f"GPU      : {torch.cuda.get_device_name(0) if GPU_AVAILABLE else 'CPU only'}")
print(f"Thresholds: accuracy ≥ {ACCURACY_THRESHOLD}, recall ≥ {RECALL_THRESHOLD}")

In [ ]:
# ============================================================================
# 3. Load Test Data & Label Map
# ============================================================================

# --- Test CSV ---
if not TEST_CSV.exists():
    raise FileNotFoundError(f"Test CSV not found at {TEST_CSV}")

df = pd.read_csv(TEST_CSV)
print(f"Test examples : {len(df)}")
print(f"Languages     : {df['language'].value_counts().to_dict()}")

# --- Label map ---
label_map_path = MODEL_DIR / "label_map.json"
if label_map_path.exists():
    with open(label_map_path) as f:
        label_map_str = json.load(f)
    label_map = {int(k): v for k, v in label_map_str.items()}
    print(f"Label map     : {label_map}")
else:
    print("⚠ label_map.json not found — using default order")
    label_map = {i: lbl for i, lbl in enumerate(LABELS)}

# label → id
label2id = {v: k for k, v in label_map.items()}
labels_sorted = sorted(label2id.keys())

# Show a few test examples
print("\n📋 Sample test rows:")
for i in range(min(5, len(df))):
    row = df.iloc[i]
    print(f"  [{row['language']}] {row['text'][:80]}…  →  {row['label']}")

In [ ]:
# ============================================================================
# 4. Load Model Pipeline
# ============================================================================

if not (MODEL_DIR / "config.json").exists():
    raise FileNotFoundError(
        f"config.json not found in {MODEL_DIR}.\n"
        f"Files present: {sorted(f.name for f in MODEL_DIR.iterdir()) if MODEL_DIR.exists() else 'dir not found'}"
    )

print(f"📥 Loading pipeline from {MODEL_DIR} …")
pipe = pipeline(
    "text-classification",
    model=str(MODEL_DIR),
    tokenizer=str(MODEL_DIR),
    top_k=None,   # return all 5 scores
    device=DEVICE,
)
print("   Pipeline loaded.")

In [ ]:
# ============================================================================
# 5. Run Inference on Test Set
# ============================================================================

def parse_top_intent(pipeline_result):
    """Extract the highest-scoring label index from pipeline output."""
    if not pipeline_result or not pipeline_result[0]:
        return 4, 0.0  # fallback
    results = pipeline_result[0] if isinstance(pipeline_result, list) else pipeline_result
    best = max(results, key=lambda x: x["score"])
    label_str = best["label"]
    if label_str.startswith("LABEL_"):
        return int(label_str.split("_")[1]), round(best["score"], 4)
    return int(label_str), round(best["score"], 4)


y_true, y_pred = [], []
errors = 0

print(f"Running inference on {len(df)} examples …")
for i, row in df.iterrows():
    text = row["text"]
    true_label = row["label"]
    true_id = label2id.get(true_label, 4)
    y_true.append(true_id)

    try:
        result = pipe(text)
        pred_id, _ = parse_top_intent(result)
        y_pred.append(pred_id)
    except Exception as exc:
        errors += 1
        y_pred.append(4)

    if (i + 1) % 25 == 0:
        print(f"   {i+1}/{len(df)} …", end="\r")

print(f"   Done. {len(y_true)} predictions  ({errors} errors)")

In [ ]:
# ============================================================================
# 6. Overall Accuracy
# ============================================================================

overall_acc = accuracy_score(y_true, y_pred)
print("=" * 56)
print(f"  📊 Overall Accuracy : {overall_acc:.4f}  (threshold: ≥{ACCURACY_THRESHOLD:.2f})")
print("=" * 56)
if overall_acc >= ACCURACY_THRESHOLD:
    print("  ✅ PASSED")
else:
    print(f"  ❌ BELOW {ACCURACY_THRESHOLD:.0%} — may need more data or epochs")

# Classification report
print("\n" + classification_report(
    y_true, y_pred,
    target_names=labels_sorted,
    zero_division=0,
))

In [ ]:
# ============================================================================
# 7. Per-Language Accuracy
# ============================================================================

print("\n--- Per-Language Accuracy ---")
all_lang_ok = True
for lang_code, lang_name in LANGS_FULL.items():
    mask = df["language"] == lang_code
    if not mask.any():
        print(f"  {lang_name:20s} : (no examples)")
        continue
    idxs = [i for i, m in enumerate(mask) if m]
    lang_y_true = [y_true[i] for i in idxs]
    lang_y_pred = [y_pred[i] for i in idxs]
    lang_acc = accuracy_score(lang_y_true, lang_y_pred)
    flag = "✅" if lang_acc >= ACCURACY_THRESHOLD else "⚠️ BELOW"
    print(f"  {lang_name:20s} : {lang_acc:.4f}  {flag}")
    if lang_acc < ACCURACY_THRESHOLD:
        all_lang_ok = False

In [ ]:
# ============================================================================
# 8. Per-Intent Recall
# ============================================================================

print("\n--- Per-Intent Recall ---")
all_intent_ok = True
recall_report = {}

for intent in labels_sorted:
    intent_id = label2id[intent]
    yt_bin = [1 if x == intent_id else 0 for x in y_true]
    yp_bin = [1 if x == intent_id else 0 for x in y_pred]

    if sum(yt_bin) == 0:
        print(f"  {intent:25s} : (no examples)")
        continue

    rec = recall_score(yt_bin, yp_bin, zero_division=0)
    recall_report[intent] = rec
    flag = "✅" if rec >= RECALL_THRESHOLD else "⚠️ BELOW"
    print(f"  {intent:25s} : {rec:.4f}  {flag}")
    if rec < RECALL_THRESHOLD:
        all_intent_ok = False

if not all_intent_ok:
    flagged = [i for i, r in recall_report.items() if r < RECALL_THRESHOLD]
    print(f"\n  ⚠️ Intents below {RECALL_THRESHOLD:.0%} recall: {', '.join(flagged)}")
    print("  → Add more training examples for these intents.")

In [ ]:
# ============================================================================
# 9. Per-Language Confusion Matrices
# ============================================================================

print("\n--- Confusion Matrices ---")
label_ids = list(range(len(labels_sorted)))

for lang_code, lang_name in LANGS_FULL.items():
    mask = df["language"] == lang_code
    if not mask.any():
        continue

    idxs = [i for i, m in enumerate(mask) if m]
    lang_y_true = [y_true[i] for i in idxs]
    lang_y_pred = [y_pred[i] for i in idxs]
    cm = confusion_matrix(lang_y_true, lang_y_pred, labels=label_ids)

    print(f"\n  {lang_name} ({len(idxs)} examples):")
    # Print formatted matrix
    header = "           " + " ".join(f"{lbl[:6]:>6}" for lbl in labels_sorted)
    print(f"  {header}")
    for i, lbl in enumerate(labels_sorted):
        row = "  ".join(f"{v:>6}" for v in cm[i])
        print(f"  {lbl:10s} {row}")

In [ ]:
# ============================================================================
# 10. Summary
# ============================================================================

print()
print("=" * 56)
checks = [
    ("Overall accuracy", overall_acc >= ACCURACY_THRESHOLD, f"{overall_acc:.4f} ≥ {ACCURACY_THRESHOLD}"),
    ("Per-language accuracy", all_lang_ok, f"All ≥ {ACCURACY_THRESHOLD}"),
    ("Per-intent recall", all_intent_ok, f"All ≥ {RECALL_THRESHOLD}"),
]

all_pass = True
for name, ok, detail in checks:
    if ok:
        print(f"  ✅ {name}: {detail}")
    else:
        all_pass = False
        print(f"  ❌ {name}: {detail}")

print("=" * 56)
if all_pass:
    print("\n🎉 All thresholds passed! Model is ready for deployment.")
else:
    print("\n⚠️ Some checks failed. Consider:")
    print("   - Training for more epochs")
    print("   - Adding more examples for low-recall intents")
    print("   - Increasing synthetic data generation")